In [2]:
import ROOT
import math
import os
import pandas as pd

%jsroot on

# ============================================================
# USER SETTINGS
# ============================================================

file_path = "/root/geant4/detector/Lung_ICRP/lung.root"
tree_name = "t"

selected_volume = 4
selected_pdg = 22             # gamma

E0 = 662.0                    # incident energy, keV

# /gps/direction 0 -1 0
incident_direction = (0.0, -1.0, 0.0)

# Histogram binning
n_theta_bins = 90             # 2-degree bins
n_energy_bins = 140           # 5-keV bins

theta_min = 0.0
theta_max = 180.0
energy_min = 0.0
energy_max = 700.0

# Output files
event_csv_file = "vlm4_gamma_entry_events.csv"
summary_csv_file = "vlm4_gamma_entry_angle_summary.csv"


# ============================================================
# OPEN ROOT FILE
# ============================================================

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"ROOT file not found:\n{file_path}"
    )

root_file = ROOT.TFile.Open(file_path)

if not root_file or root_file.IsZombie():
    raise RuntimeError(
        f"Could not open ROOT file:\n{file_path}"
    )

tree = root_file.Get(tree_name)

if not tree:
    root_file.ls()
    raise RuntimeError(
        f"Tree '{tree_name}' was not found."
    )

print("ROOT file opened successfully")
print("Tree entries:", tree.GetEntries())


# ============================================================
# CHECK REQUIRED BRANCHES
# ============================================================

# et is optional because it has a different multiplicity.
required_branches = [
    "pdg",
    "vlm",
    "trk",
    "stp",
    "k",
    "px",
    "py",
    "pz"
]

available_branches = {
    branch.GetName()
    for branch in tree.GetListOfBranches()
}

missing_branches = [
    name for name in required_branches
    if name not in available_branches
]

if missing_branches:
    raise RuntimeError(
        "Missing required branches: "
        + ", ".join(missing_branches)
    )

has_et_branch = "et" in available_branches

print("Required branches are available.")
print("et branch available:", has_et_branch)


# ============================================================
# NORMALIZE INCIDENT DIRECTION
# ============================================================

ix, iy, iz = incident_direction

incident_norm = math.sqrt(
    ix**2 + iy**2 + iz**2
)

if incident_norm <= 0:
    raise ValueError(
        "Incident direction cannot be zero."
    )

ix /= incident_norm
iy /= incident_norm
iz /= incident_norm


# ============================================================
# REMOVE OLD ROOT OBJECTS
# ============================================================

for object_name in [
    "h_theta_Eprime",
    "h_theta_transfer",
    "h_theta_et",
    "c_vlm4_gamma"
]:
    old_object = ROOT.gROOT.FindObject(object_name)

    if old_object:
        if old_object.InheritsFrom("TCanvas"):
            old_object.Close()
        else:
            old_object.Delete()


# ============================================================
# CREATE HISTOGRAMS
# ============================================================

# Plot 1: theta versus scattered photon energy E' = k
h_theta_Eprime = ROOT.TH2F(
    "h_theta_Eprime",
    "Gamma entry direction versus scattered energy E', volume 4;"
    "Angle #theta relative to incident beam (degrees);"
    "E' = k (keV);"
    "Counts per bin",
    n_theta_bins, theta_min, theta_max,
    n_energy_bins, energy_min, energy_max
)

# Plot 2: theta versus energy transferred E0-E'
h_theta_transfer = ROOT.TH2F(
    "h_theta_transfer",
    "Gamma entry direction versus E_{0}-E', volume 4;"
    "Angle #theta relative to incident beam (degrees);"
    "E_{0}-E' (keV);"
    "Counts per bin",
    n_theta_bins, theta_min, theta_max,
    n_energy_bins, energy_min, energy_max
)

# Plot 3: theta versus stored et, only where et[i] exists
h_theta_et = ROOT.TH2F(
    "h_theta_et",
    "Gamma entry direction versus stored et, volume 4;"
    "Angle #theta relative to incident beam (degrees);"
    "Stored et (keV);"
    "Counts per bin",
    n_theta_bins, theta_min, theta_max,
    n_energy_bins, energy_min, energy_max
)

for histogram in [
    h_theta_Eprime,
    h_theta_transfer,
    h_theta_et
]:
    histogram.SetStats(0)


# ============================================================
# EVENT LOOP
# ============================================================

rows = []

all_volume4_gamma_records = 0
unique_entering_gamma_tracks = 0
zero_momentum_records = 0
invalid_records = 0
entries_with_et = 0
entries_without_et = 0

for event_number, event in enumerate(tree):

    # Do not include et here.
    # et is shorter and would incorrectly truncate this loop.
    n_core = min(
        len(event.pdg),
        len(event.vlm),
        len(event.trk),
        len(event.stp),
        len(event.k),
        len(event.px),
        len(event.py),
        len(event.pz)
    )

    # Tracks already recorded in volume 4 in this event.
    counted_tracks = set()

    for i in range(n_core):

        if int(event.vlm[i]) != selected_volume:
            continue

        if int(event.pdg[i]) != selected_pdg:
            continue

        all_volume4_gamma_records += 1

        track_id = int(event.trk[i])

        # Keep only first stored vlm=4 record for each gamma track.
        if track_id in counted_tracks:
            continue

        counted_tracks.add(track_id)

        px = float(event.px[i])
        py = float(event.py[i])
        pz = float(event.pz[i])

        momentum = math.sqrt(
            px**2 + py**2 + pz**2
        )

        if momentum <= 0:
            zero_momentum_records += 1
            continue

        # Unit vector along gamma direction in volume 4.
        ux = px / momentum
        uy = py / momentum
        uz = pz / momentum

        # Angle relative to the original incident beam.
        cos_theta = (
            ix * ux +
            iy * uy +
            iz * uz
        )

        cos_theta = max(
            -1.0,
            min(1.0, cos_theta)
        )

        theta_deg = math.degrees(
            math.acos(cos_theta)
        )

        # For a photon, k is the photon energy.
        Eprime_keV = float(event.k[i])

        # Energy lost/transferred before reaching volume 4.
        transfer_keV = E0 - Eprime_keV

        if not (
            math.isfinite(theta_deg)
            and math.isfinite(Eprime_keV)
            and math.isfinite(transfer_keV)
        ):
            invalid_records += 1
            continue

        # et is optional because it may not have an element at i.
        et_value = float("nan")

        if has_et_branch and i < len(event.et):
            candidate_et = float(event.et[i])

            if math.isfinite(candidate_et):
                et_value = candidate_et
                entries_with_et += 1
                h_theta_et.Fill(theta_deg, et_value)
            else:
                entries_without_et += 1
        else:
            entries_without_et += 1

        h_theta_Eprime.Fill(
            theta_deg,
            Eprime_keV
        )

        h_theta_transfer.Fill(
            theta_deg,
            transfer_keV
        )

        rows.append({
            "event_number": event_number,
            "track_id": track_id,
            "record_index": i,
            "step_number_at_first_vlm4_record":
                int(event.stp[i]),
            "vlm": int(event.vlm[i]),
            "pdg": int(event.pdg[i]),
            "theta_deg": theta_deg,
            "Eprime_k_keV": Eprime_keV,
            "E0_minus_Eprime_keV": transfer_keV,
            "stored_et_keV": et_value,
            "px": px,
            "py": py,
            "pz": pz
        })

        unique_entering_gamma_tracks += 1


# ============================================================
# DATAFRAME AND RESULTS
# ============================================================

df = pd.DataFrame(rows)

print("\nSelection:")
print("vlm =", selected_volume)
print("pdg =", selected_pdg)
print("No process selection")

print("\nAll vlm=4 gamma step records:",
      all_volume4_gamma_records)

print("Unique gamma tracks entering vlm=4:",
      unique_entering_gamma_tracks)

print("Zero-momentum first records:",
      zero_momentum_records)

print("Invalid records:",
      invalid_records)

print("Entries with usable et:",
      entries_with_et)

print("Entries without corresponding et:",
      entries_without_et)

if df.empty:
    raise RuntimeError(
        "No valid gamma entries were found in volume 4."
    )


# ============================================================
# ANGLE-BINNED SUMMARY
# ============================================================

theta_bin_width = (
    theta_max - theta_min
) / n_theta_bins

df["theta_bin_left_deg"] = (
    theta_min
    + (
        (df["theta_deg"] - theta_min)
        // theta_bin_width
    ) * theta_bin_width
)

df["theta_bin_right_deg"] = (
    df["theta_bin_left_deg"]
    + theta_bin_width
)

df["theta_bin_center_deg"] = (
    df["theta_bin_left_deg"]
    + theta_bin_width / 2.0
)

summary_df = (
    df.groupby(
        [
            "theta_bin_left_deg",
            "theta_bin_right_deg",
            "theta_bin_center_deg"
        ],
        as_index=False
    )
    .agg(
        counts=("theta_deg", "size"),
        mean_Eprime_keV=("Eprime_k_keV", "mean"),
        std_Eprime_keV=("Eprime_k_keV", "std"),
        mean_E0_minus_Eprime_keV=(
            "E0_minus_Eprime_keV",
            "mean"
        ),
        std_E0_minus_Eprime_keV=(
            "E0_minus_Eprime_keV",
            "std"
        ),
        mean_stored_et_keV=("stored_et_keV", "mean"),
        std_stored_et_keV=("stored_et_keV", "std")
    )
    .sort_values("theta_bin_center_deg")
)

# Limit long decimal values.
df = df.round(6)
summary_df = summary_df.round(6)


# ============================================================
# SAVE CSV FILES
# ============================================================

df.to_csv(
    event_csv_file,
    index=False
)

summary_df.to_csv(
    summary_csv_file,
    index=False
)

print("\nCSV files saved:")
print(event_csv_file)
print(summary_csv_file)


# ============================================================
# DRAW THREE PLOTS
# ============================================================

c_vlm4_gamma = ROOT.TCanvas(
    "c_vlm4_gamma",
    "Gamma entry into volume 4",
    1600,
    1100
)

c_vlm4_gamma.Divide(2, 2)


# ------------------------------------------------------------
# Plot 1: theta versus E'
# ------------------------------------------------------------

pad1 = c_vlm4_gamma.cd(1)

pad1.SetLeftMargin(0.11)
pad1.SetRightMargin(0.15)
pad1.SetBottomMargin(0.12)

h_theta_Eprime.Draw("COLZ")


# ------------------------------------------------------------
# Plot 2: theta versus E0-E'
# ------------------------------------------------------------

pad2 = c_vlm4_gamma.cd(2)

pad2.SetLeftMargin(0.11)
pad2.SetRightMargin(0.15)
pad2.SetBottomMargin(0.12)

h_theta_transfer.Draw("COLZ")


# ------------------------------------------------------------
# Plot 3: theta versus stored et
# ------------------------------------------------------------

pad3 = c_vlm4_gamma.cd(3)

pad3.SetLeftMargin(0.11)
pad3.SetRightMargin(0.15)
pad3.SetBottomMargin(0.12)

if h_theta_et.GetEntries() > 0:
    h_theta_et.Draw("COLZ")
else:
    empty_text = ROOT.TPaveText(
        0.15, 0.35, 0.85, 0.65, "NDC"
    )
    empty_text.SetFillStyle(0)
    empty_text.SetBorderSize(0)
    empty_text.SetTextSize(0.05)
    empty_text.AddText(
        "No index-aligned et values for vlm=4 gamma entries"
    )
    empty_text.Draw()


# ------------------------------------------------------------
# Plot 4: entry-energy spectrum
# ------------------------------------------------------------

pad4 = c_vlm4_gamma.cd(4)

pad4.SetLeftMargin(0.12)
pad4.SetRightMargin(0.05)
pad4.SetBottomMargin(0.12)

h_entry_energy = ROOT.TH1F(
    "h_entry_energy",
    "Gamma entry-energy spectrum, volume 4;"
    "E' = k (keV);"
    "Entering gamma tracks",
    n_energy_bins,
    energy_min,
    energy_max
)

for value in df["Eprime_k_keV"]:
    h_entry_energy.Fill(float(value))

h_entry_energy.SetStats(1)
h_entry_energy.Draw("HIST")


# ============================================================
# DISPLAY
# ============================================================

c_vlm4_gamma.Modified()
c_vlm4_gamma.Update()
c_vlm4_gamma.Draw()

ROOT file opened successfully
Tree entries: 1000000
Required branches are available.
et branch available: True

Selection:
vlm = 4
pdg = 22
No process selection

All vlm=4 gamma step records: 2545060
Unique gamma tracks entering vlm=4: 938363
Zero-momentum first records: 0
Invalid records: 0
Entries with usable et: 0
Entries without corresponding et: 938363

CSV files saved:
vlm4_gamma_entry_events.csv
vlm4_gamma_entry_angle_summary.csv
